# Auditoría de registros y muestras de entrenamiento

**Objetivo:** Trazabilidad de registros históricos y muestras de entrenamiento.

Este cuaderno contiene el código completo del experimento `audit_pipeline.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `incendios_coordenadas_gee.csv`, `dataset_entrenamiento_sierra_realista.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'auditoria'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['incendios_coordenadas_gee.csv', 'dataset_entrenamiento_sierra_realista.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
BASE_DIR = str(OUTPUT_DIR)
OUT_DIR = str(OUTPUT_DIR)
FILE_RAW = str(DATA_FILES['incendios_coordenadas_gee.csv'])
FILE_ML = str(DATA_FILES['dataset_entrenamiento_sierra_realista.csv'])


### Definición: `es_sierra`
Esta función auxiliar forma parte del protocolo histórico; revisa sus reglas de selección antes de cambiar la población analizada.

In [ ]:
def es_sierra(prov_val):
    if not isinstance(prov_val, str): 
        return False
    p_clean = prov_val.strip().strip("'").lower()
    es_prov_sierra = any(k in p_clean for k in ['pichincha', 'azuay', 'imbabura', 'loja', 'tungurahua', 'chimborazo', 'ca', 'cotopaxi', 'carchi', 'bol'])
    es_prov_otra = any(k in p_clean for k in ['pastaza', 'orellana', 'galapagos', 'esmeraldas', 'los rios', 'el oro', 'guayas', 'manabi', 'santa elena', 'santo domingo', 'morona', 'zamora', 'napo', 'sucumbi'])
    return es_prov_sierra and not es_prov_otra


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("="*80)
print("  AUDITORÍA DE TRAZABILIDAD DE REGISTROS DE INCENDIOS (SIERRA ECUATORIANA)")
print("="*80)

# 1. Base bruta original
df_raw = pd.read_csv(FILE_RAW)
n_raw = len(df_raw)
print(f"\n1. Base histórica bruta original (incendios_coordenadas_gee.csv): {n_raw:,} registros.")

# Verificar nulos en fecha y coordenadas en el dataset bruto
n_null_date = df_raw['date'].isna().sum()
n_null_lat = df_raw['latitude'].isna().sum()
n_null_lon = df_raw['longitude'].isna().sum()
print(f"   - Fechas nulas/inválidas en base bruta: {n_null_date}")
print(f"   - Latitud nula en base bruta: {n_null_lat}")
print(f"   - Longitud nula en base bruta: {n_null_lon}")

# 2. Filtro regional: Provincias de la Sierra
df_sierra = df_raw[df_raw['provincia'].apply(es_sierra)].copy()
n_sierra = len(df_sierra)
n_fuera_sierra = n_raw - n_sierra
print(f"\n2. Filtro por las 10 provincias de la Sierra Ecuatoriana: {n_sierra:,} registros.")
print(f"   - Registros pertenecientes a Costa/Amazonía/Galápagos descartados: {n_fuera_sierra:,}")

# 3. Verificación de coordenadas y fechas en la Sierra
n_sierra_null_coords = df_sierra['latitude'].isna().sum() + df_sierra['longitude'].isna().sum()


### Etapa 2: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print(f"   - Coordenadas nulas dentro de la Sierra: {n_sierra_null_coords}")

# 4. Depuración de duplicados espaciales exactos (misma latitud y longitud)
df_sierra_unique_spatial = df_sierra.drop_duplicates(subset=['latitude', 'longitude']).copy()
n_unique_spatial = len(df_sierra_unique_spatial)
n_duplicados_espaciales = n_sierra - n_unique_spatial
print(f"\n3. Depuración de duplicados espaciales exactos (.drop_duplicates(['latitude', 'longitude'])): {n_unique_spatial:,} registros únicos.")
print(f"   - Duplicados espaciales eliminados (mismo punto geográfico con múltiples fechas o reportes): {n_duplicados_espaciales:,}")

# 5. Muestreo / Extracción en GEE dentro del límite andino (pajonales/valles) y filtrado de nulos
df_ml = pd.read_csv(FILE_ML)
df_ml_presencias = df_ml[df_ml['clase'] == 1].copy()
n_ml_presencias = len(df_ml_presencias)
n_gee_descartados = n_unique_spatial - n_ml_presencias

print(f"\n4. Extracción de covariables en GEE + Máscara espacial andina + Filtrado de nulos en Sentinel-2/SRTM: {n_ml_presencias:,} presencias.")
print(f"   - Puntos fuera de la cobertura útil de Sentinel-2/SRTM o fuera del buffer andino eliminados en GEE: {n_gee_descartados:,}")

# Cifras finales ML
n_ml_ausencias = len(df_ml[df_ml['clase'] == 0])
n_ml_total = len(df_ml)

print("\n" + "="*80)
print("  RESUMEN DE TRAZABILIDAD EXACTA Y DESGLOSE")
print("="*80)
print(f"Base histórica bruta nacional:                     {n_raw:>7,}")


### Etapa 3: Inspección y cálculo intermedio
Ejecuta esta etapa después de la anterior y revisa los valores impresos. Las variables quedan disponibles para explorarlas o modificarlas en otra celda.

In [ ]:
print(f" (-) Descarte por provincias (Costa/Amazonía/Galápagos): -{n_fuera_sierra:>6,}")
print(f" Base histórica regional (Sierra Ecuatoriana):     {n_sierra:>7,}")
print(f" (-) Duplicados espaciales exactos (mismo punto lat/lon): -{n_duplicados_espaciales:>6,}")
print(f" Presencias únicas geoespaciales en la Sierra:     {n_unique_spatial:>7,}")
print(f" (-) Descarte en GEE (cobertura nubes/Sentinel-2 nulos):  -{n_gee_descartados:>6,}")
print(f" Presencias finales en dataset de ML (clase=1):    {n_ml_presencias:>7,}")
print(f" (+) Pseudoausencias difíciles muestreadas (clase=0):+{n_ml_ausencias:>6,}")
print(f" Total observaciones dataset de ML (N_total):       {n_ml_total:>7,}")
print("="*80)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
